---
# Part 1 — Clustering

In [11]:
import subprocess, sys

In [12]:
import time
import math
import random
import string
import os
import sys
from collections import defaultdict
import numpy as np
tmp = r'C:\spark-tmp'
os.environ['SPARK_LOCAL_DIRS'] = 'C:/spark-tmp'
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
from pyspark.mllib.linalg import Vectors, Vector

#conf = SparkConf().setAppName('M25DE1058_Assignment4').setMaster('local[*]')
conf = (SparkConf()
    .setAppName('CSL7110_Assignment4')
    .setMaster('local[*]')
    .set('spark.local.dir',          tmp)
    .set('spark.driver.extraJavaOptions',
         f'-Djava.io.tmpdir={tmp}')          # force JVM-level tmpdir
    .set('spark.executor.extraJavaOptions',
         f'-Djava.io.tmpdir={tmp}'))
sc = SparkContext(conf=conf)
#sc = SparkContext.getOrCreate(conf=conf)
sc.setLogLevel('ERROR')


###  `readVectorsSeq`

In [13]:
def readVectorsSeq(filename):
    vectors = []
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            values = list(map(float, line.split(',')))
            vectors.append(Vectors.dense(values))
    return vectors

print('readVectorsSeq function defined successfully')

readVectorsSeq function defined successfully


In [5]:
def sq_dist(a, b):
    return Vectors.squared_distance(a, b)

###  `kcenter`

In [6]:
def kcenter(P, k):
    assert k <= len(P), 'k must be <= number of points'

    n = len(P)

    # Pick the first center
    centers = [P[0]]

    min_dist = [sq_dist(P[i], P[0]) for i in range(n)]  
    for _ in range(k - 1):                             
        
        next_idx = max(range(n), key=lambda i: min_dist[i]) 
        new_center = P[next_idx]
        centers.append(new_center)

        for i in range(n):
            d = sq_dist(P[i], new_center)
            if d < min_dist[i]:
                min_dist[i] = d
    
    return centers

print('kcenter function defined successfully.')

kcenter function defined successfully.


### `kmeansPP`

In [7]:
def kmeansPP(P, k):
   
    assert k <= len(P), 'k must be <= number of points'

    n = len(P)
    random.seed(42)  
    first_idx = random.randint(0, n - 1)
    centers = [P[first_idx]]
    min_dist = [sq_dist(P[i], centers[0]) for i in range(n)]  

    for _ in range(k - 1):                                  
        total = sum(min_dist)                                 

        if total == 0:
            next_idx = random.randint(0, n - 1)
        else:
            threshold = random.uniform(0, total)
            cumulative = 0.0
            next_idx = n - 1  
            for i in range(n):                                  
                cumulative += min_dist[i]
                if cumulative >= threshold:
                    next_idx = i
                    break

        new_center = P[next_idx]
        centers.append(new_center)

        for i in range(n):
            d = sq_dist(P[i], new_center)
            if d < min_dist[i]:
                min_dist[i] = d

    return centers

print('kmeansPP function defined successfully.')

kmeansPP function defined successfully.


### `kmeansObj` 

In [8]:
def kmeansObj(P, C):
    total = 0.0
    for p in P:
        
        min_d = min(sq_dist(p, c) for c in C)
        total += min_d
    return total / len(P)

print('kmeansObj function defined successfully.')

kmeansObj function defined successfully.


In [11]:
DATASET_PATH = 'spambase.data'   
k  = 100
k1 = 400

raw_vectors = readVectorsSeq(DATASET_PATH)
print(f'Loaded {len(raw_vectors)} points with {len(raw_vectors[0])} dimensions.')
P = raw_vectors 

print(f'Using {len(P)} points, k={k}, k1={k1}')

Loaded 4601 points with 58 dimensions.
Using 4601 points, k=100, k1=400


In [12]:
#kcenter(P, k) — with timing
start = time.time()
C_kcenter = kcenter(P, k)
elapsed = time.time() - start

print(f'kcenter running time : {elapsed:.4f} seconds')

kcenter running time : 1.8868 seconds


In [13]:
# kmeansPP(P, k) 

start = time.time()
C_kmeanspp = kmeansPP(P, k)
elapsed_kpp = time.time() - start

obj2 = kmeansObj(P, C_kmeanspp)

#print(f'Set of k centres: {C_kmeanspp}')
print(f'kmeansPP running time: {elapsed_kpp:.4f} seconds')
print(f'k-means objective Avg Squared distance    : {obj2:.6f}')

kmeansPP running time: 1.8429 seconds
k-means objective Avg Squared distance    : 834.417541


In [14]:
#kcenter(P, k1) -> kmeansPP(X, k) -> kmeansObj(P, C)
start = time.time()
X = kcenter(P, k1)                  
C_combined = kmeansPP(X, k)       
obj3 = kmeansObj(P, C_combined)
print(f'k-means objective Avg Squared distance with k1 centres : {obj3:.6f}')
print(f'k-means objective Avg Squared distance with k centres  : {obj2:.6f}')

k-means objective Avg Squared distance with k1 centres : 1714.500991
k-means objective Avg Squared distance with k centres  : 834.417541


From the above output, k1 from the kcenter which is greater than k acts as a coreset, which is a representation of the entire dataset P. But it is not optimized for squared distance. So, kcenter spreads the centers for outliers, resulting in poor average squared distance. 
On the other hand, Kmeans++ is optimized for squared distance. So even with k centers, it produces a much better average squared distance. 


# Part 2 — Web Search / Inverted Index

In [15]:
STOP_WORDS = {
    'a', 'an', 'the', 'they', 'these', 'this', 'for',
    'is', 'are', 'was', 'of', 'or', 'and', 'does', 'will', 'whose'
}
PUNCTUATION = set('{}[]<>=(). ,;\'\'?#!-:')
def normalise_word(word):
    word = word.lower()
    if word.endswith('s') and len(word) > 1:
        word = word[:-1]
    return word
def tokenise(text):
    cleaned = ''.join(' ' if ch in PUNCTUATION else ch for ch in text)
    tokens = []
    for raw in cleaned.split():
        lower = raw.lower()
        tokens.append(lower)
    return tokens


### `MySet`

In [16]:
class MySet:
    def __init__(self):
        self._data = set()

    def addElement(self, element):
        self._data.add(element)

    def union(self, otherSet):
        result = MySet()
        result._data = self._data | otherSet._data
        return result

    def intersection(self, otherSet):
        result = MySet()
        result._data = self._data & otherSet._data
        return result

    def __iter__(self):
        return iter(self._data)

    def __len__(self):
        return len(self._data)

    def __contains__(self, item):
        return item in self._data

    def __repr__(self):
        return f'MySet({self._data})'

print('MySet defined successfully!')

MySet defined successfully!


### `Position`

In [17]:
class Position:
   
    def __init__(self, page_entry, word_index):
        self._page_entry = page_entry
        self._word_index = word_index

    def getPageEntry(self):
        return self._page_entry

    def getWordIndex(self):
        return self._word_index

    def __repr__(self):
        return f'Position({self._page_entry.pageName}, {self._word_index})'

    def __hash__(self):
        return hash((self._page_entry.pageName, self._word_index))

    def __eq__(self, other):
        return (self._page_entry.pageName == other._page_entry.pageName
                and self._word_index == other._word_index)

print('Position defined successfully !')

Position defined successfully !


### `WordEntry`

In [18]:
class WordEntry:
    def __init__(self, word):
        self._word = word
        self._positions = MySet()

    def addPosition(self, position):
        self._positions.addElement(position)

    def addPositions(self, positions):
        for pos in positions:
            self._positions.addElement(pos)

    def getAllPositionsForThisWord(self):
        return list(self._positions)

    def getTermFrequency(self, page_entry):
        count = sum(
            1 for pos in self._positions
            if pos.getPageEntry().pageName == page_entry.pageName
        )
        total_words = page_entry.totalWords
        if total_words == 0:
            return 0.0
        return count / total_words

    def getWord(self):
        return self._word

    def __repr__(self):
        return f'WordEntry({self._word}, positions={len(self._positions)})'

print('WordEntry defined successfully!')

WordEntry defined successfully!


### `PageIndex`

In [19]:
class PageIndex:
    
    def __init__(self):
        self._index = {}

    def addPositionForWord(self, word_str, position):
        if word_str not in self._index:
            self._index[word_str] = WordEntry(word_str)
        self._index[word_str].addPosition(position)

    def getWordEntries(self):
        return list(self._index.values())

    def getWordEntry(self, word_str):
        return self._index.get(word_str)

    def __repr__(self):
        return f'PageIndex({list(self._index.keys())})'

print('PageIndex defined successfully!')

PageIndex defined successfully!


### `PageEntry`

In [20]:
class PageEntry:
    WEBPAGES_FOLDER = 'webpages'

    def __init__(self, pageName):
        self.pageName = pageName
        self._pageIndex = PageIndex()
        self.totalWords = 0  # total number of tokens (including stop words)
        self._buildIndex()

    def _buildIndex(self):
        filepath = os.path.join(PageEntry.WEBPAGES_FOLDER, self.pageName)
        try:
            with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
                text = f.read()
        except FileNotFoundError:
            print(f'Warning: file not found — {filepath}')
            return

        tokens = tokenise(text)  
        self.totalWords = len(tokens)  

        for word_idx, token in enumerate(tokens, start=1):
            if token in STOP_WORDS or not token:
                continue 
            normalised = normalise_word(token)
            if not normalised:
                continue
            pos = Position(self, word_idx)
            self._pageIndex.addPositionForWord(normalised, pos)

    def getPageIndex(self):
        return self._pageIndex

    def __repr__(self):
        return f'PageEntry({self.pageName})'

    def __hash__(self):
        return hash(self.pageName)

    def __eq__(self, other):
        return self.pageName == other.pageName

print('PageEntry defined successfully!')

PageEntry defined successfully!


### `MyHashTable`

In [21]:
class MyHashTable:
   
    DEFAULT_SIZE = 1024 

    def __init__(self, size=None):
        self._size = size or MyHashTable.DEFAULT_SIZE
        self._buckets = [[] for _ in range(self._size)]

    def getHashIndex(self, word_str):
        return hash(word_str) % self._size

    def _find(self, word_str):
        idx = self.getHashIndex(word_str)
        for w, entry in self._buckets[idx]:
            if w == word_str:
                return entry
        return None

    def addPositionsForWord(self, word_entry):
        word = word_entry.getWord()
        idx = self.getHashIndex(word)
        existing = self._find(word)
        if existing is None:
            self._buckets[idx].append((word, word_entry))
        else:
            existing.addPositions(word_entry.getAllPositionsForThisWord())

    def getWordEntry(self, word_str):
        return self._find(word_str)

    def getAllWordEntries(self):
        result = []
        for bucket in self._buckets:
            for _, entry in bucket:
                result.append(entry)
        return result

    def __repr__(self):
        n = sum(len(b) for b in self._buckets)
        return f'MyHashTable(size={self._size}, entries={n})'

print('MyHashTable defined successfully!')

MyHashTable defined successfully!


### `InvertedPageIndex`

In [22]:
class InvertedPageIndex:

    def __init__(self):
        self._table = MyHashTable()
        self._pages = {}   

    def addPage(self, page_entry):
        self._pages[page_entry.pageName] = page_entry

        for word_entry in page_entry.getPageIndex().getWordEntries():
            self._table.addPositionsForWord(word_entry)

    def getPagesWhichContainWord(self, word_str):
        norm = normalise_word(word_str.lower())
        result = MySet()
        entry = self._table.getWordEntry(norm)
        if entry is None:
            return result
        for pos in entry.getAllPositionsForThisWord():
            result.addElement(pos.getPageEntry())
        return result

    def getWordEntry(self, word_str):
        norm = normalise_word(word_str.lower())
        return self._table.getWordEntry(norm)

    def getPageEntry(self, pageName):
        return self._pages.get(pageName)

    def totalPages(self):
        return len(self._pages)

    def __repr__(self):
        return f'InvertedPageIndex(pages={len(self._pages)})'

print('InvertedPageIndex defined successfully!')

InvertedPageIndex defined successfully!


###  `SearchEngine`

In [23]:
class SearchEngine:
    def __init__(self):
        self._index = InvertedPageIndex()

    def _tfidf(self, word_str, page_entry):
        norm = normalise_word(word_str.lower())
        N = self._index.totalPages()

        we = self._index.getWordEntry(norm)
        if we is None or N == 0:
            return 0.0

        pages_with_word = self._index.getPagesWhichContainWord(norm)
        n_w = len(pages_with_word)
        if n_w == 0:
            return 0.0

        count_in_page = sum(
            1 for pos in we.getAllPositionsForThisWord()
            if pos.getPageEntry().pageName == page_entry.pageName
        )
        if page_entry.totalWords == 0:
            return 0.0
        tf = count_in_page / page_entry.totalWords

        idf = math.log(N / n_w)

        return tf * idf

    def _action_addPage(self, page_name):
        pe = PageEntry(page_name)
        self._index.addPage(pe)

    def _action_queryFindPagesWhichContainWord(self, word):
        pages = self._index.getPagesWhichContainWord(word)
        if len(pages) == 0:
            print(f'No webpage contains word {word}')
        else:
            names = sorted(pe.pageName for pe in pages)
            print(', '.join(names))

    def _action_queryFindPositionsOfWordInAPage(self, word, page_name):
        pe = self._index.getPageEntry(page_name)
        if pe is None:
            print(f'No webpage {page_name} found')
            return

        norm = normalise_word(word.lower())
        we = self._index.getWordEntry(norm)
        if we is None:
            print(f'Webpage {page_name} does not contain word {word}')
            return

        positions = [
            pos.getWordIndex()
            for pos in we.getAllPositionsForThisWord()
            if pos.getPageEntry().pageName == page_name
        ]

        if not positions:
            print(f'Webpage {page_name} does not contain word {word}')
        else:
            print(', '.join(str(p) for p in sorted(positions)))

    def performAction(self, actionMessage):
        parts = actionMessage.strip().split()
        if not parts:
            return

        action = parts[0]

        if action == 'addPage':
            if len(parts) < 2:
                print('Usage: addPage <page_name>')
                return
            self._action_addPage(parts[1])

        elif action == 'queryFindPagesWhichContainWord':
            if len(parts) < 2:
                print('Usage: queryFindPagesWhichContainWord <word>')
                return
            self._action_queryFindPagesWhichContainWord(parts[1])

        elif action == 'queryFindPositionsOfWordInAPage':
            if len(parts) < 3:
                print('Usage: queryFindPositionsOfWordInAPage <word> <page>')
                return
            self._action_queryFindPositionsOfWordInAPage(parts[1], parts[2])

        else:
            print(f'Unknown action: {action}')

print('SearchEngine defined successfully')

SearchEngine defined successfully


### Run actions from `actions.txt`

In [24]:
ACTIONS_FILE  = 'actions.txt'   
ANSWERS_FILE  = 'answers.txt'   

engine = SearchEngine()
try:
    with open(ACTIONS_FILE, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            engine.performAction(line)
except FileNotFoundError:
    print(f'actions.txt not found in the specific path')

No webpage contains word delhi
stack_datastructure_wiki
stack_datastructure_wiki
Webpage stack_datastructure_wiki does not contain word magazines
No webpage contains word allain
stack_cprogramming
stack_cprogramming
stack_cprogramming
stack_oracle
stack_cprogramming, stack_datastructure_wiki, stackoverflow
stackmagazine


In [ ]:
sc.stop()